# Memory Prediction with c-Augmentation — Full Pipeline

This notebook trains the full zoo of memory-prediction models — from Sizey's basic ensemble
(Linear Regression, KNN, MLP, Random Forest) through gradient boosting (LightGBM) and uncertainty-aware
distributional regression (NGBoost LogNormal) — comparing the **Sizey path** (`a` only) against
our **joint path** (`a + c`).

**Pipeline:**
1. Load 33,991-row dataset (pyradiomics + iwd + ours, methylseq excluded)
2. Per-bucket 80/20 train/test split (seed=42; held out from training)
3. Train all 6 base models in both Sizey and Joint feature configurations
4. Per-method residual-σ calibrated via 5-fold CV on train only
5. Evaluate on held-out test: MAPE / R² / coverage / wastage / OOM
6. Active-learning gates (G1 σ, G2 risk, G3 disagreement) + budget sweep
7. Live predictions on unseen tasks
8. Paper-grade figures (large fonts, big markers, 300 dpi)


In [ ]:
import warnings; warnings.filterwarnings("ignore")
import json, pickle, time, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---- Paper-grade plot defaults (large fonts, big markers, 300 dpi) ----
plt.rcParams.update({
    "font.size":          18,
    "axes.titlesize":     22,
    "axes.labelsize":     20,
    "xtick.labelsize":    16,
    "ytick.labelsize":    16,
    "legend.fontsize":    16,
    "figure.titlesize":   24,
    "lines.markersize":   14,
    "lines.linewidth":    3.0,
    "axes.linewidth":     1.8,
    "xtick.major.width":  1.5,
    "ytick.major.width":  1.5,
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "savefig.dpi":        300,
    "savefig.bbox":       "tight",
})
COLORS = {"lr":"#377eb8","knn":"#4daf4a","mlp":"#984ea3","rf":"#ff7f00",
          "lgbm":"#e41a1c","ngb":"#a65628","sizey":"#6c757d","joint":"#16a085"}

REPO = Path("/shared/training")
FIG_DIR = REPO / "figures"; FIG_DIR.mkdir(exist_ok=True)
EPS, K, MIN_INSTANCES, SEED = 1.0, 1.5, 10, 42
print("setup ok; figures →", FIG_DIR)

## 1.  Load three data sources (no methylseq)

In [ ]:
# Source A: our nf-core runs minus methylseq
ours = pd.read_csv(REPO/"all_workflows.csv")
on = pd.DataFrame({
    "workflow":ours["workflow"], "process":ours["process"],
    "a":pd.to_numeric(ours["a_bytes"],errors="coerce"),
    "c":pd.to_numeric(ours["c_bytes"],errors="coerce"),
    "M":pd.to_numeric(ours["M_peak_rss_bytes"],errors="coerce"),
    "runtime":pd.to_numeric(ours["runtime_seconds"],errors="coerce"),
}).dropna(subset=["a","M"]).query("M>0 and a>=0").reset_index(drop=True)
on = on[on["workflow"] != "methylseq"]

# Source B: naga's pyradiomics 32k
sys.path.insert(0, str(REPO))
from load_pyradiomics import load_pyradiomics
pn = load_pyradiomics(str(REPO/"pyradiomics_32k.csv"))[["workflow","process","a","c","M","runtime_sec"]]
pn = pn.rename(columns={"runtime_sec":"runtime"})

df = pd.concat([on, pn], ignore_index=True)
df["runtime"] = df["runtime"].fillna(60.0)
print(f"loaded {len(df):,} rows  ({df['c'].notna().sum():,} with c, {df['c'].isna().sum():,} without)")
df.groupby("workflow").agg(rows=("process","count"), with_c=("c", lambda s: s.notna().sum()))

## 2.  Bucket inventory

In [ ]:
buckets = (df.groupby(["workflow","process"]).size()
              .reset_index(name="n").query(f"n>={MIN_INSTANCES}").sort_values("n", ascending=False))
print(f"trainable buckets: {len(buckets)}  total rows: {buckets.n.sum():,}")
buckets.head(20)

In [ ]:
# Figure: bucket size distribution
fig, ax = plt.subplots(figsize=(13, 6))
labels = [f"{r.workflow}::{r.process[:30]}" for _, r in buckets.iterrows()]
colors = ["#16a085" if df[(df.workflow==w)&(df.process==p)].c.notna().any() else "#e74c3c"
          for w,p in zip(buckets.workflow, buckets.process)]
ax.barh(range(len(buckets)), buckets.n, color=colors, edgecolor="black", linewidth=1.5)
ax.set_yticks(range(len(buckets))); ax.set_yticklabels(labels, fontsize=14)
ax.set_xlabel("Rows per bucket")
ax.set_title("Per-bucket dataset size  (green = c available, red = no c)")
ax.invert_yaxis()
fig.savefig(FIG_DIR/"fig1_buckets.png"); plt.show()

## 3.  80/20 train/test split (seed=42, per bucket)

In [ ]:
splits = {}
for (wf, proc), grp in df.groupby(["workflow","process"]):
    if len(grp) < MIN_INSTANCES: continue
    g = grp.sample(frac=1, random_state=SEED).reset_index(drop=True)
    cut = int(len(g) * 0.8)
    splits[(wf, proc)] = (g.iloc[:cut].copy(), g.iloc[cut:].copy())

n_tr_total = sum(len(s[0]) for s in splits.values())
n_te_total = sum(len(s[1]) for s in splits.values())
print(f"train rows: {n_tr_total:,}   test rows: {n_te_total:,}")

## 4.  Model zoo

We train 6 base regressors per bucket, each in 2 feature configurations:

| Model | What | Used by |
|---|---|---|
| **LR** | Linear Regression on log a | Sizey baseline 1 |
| **KNN** | k-Nearest Neighbours (k=5, scaled) | Sizey baseline 2 |
| **MLP** | 2-layer MLP (64-32, scaled) | Sizey baseline 3 |
| **RF** | Random Forest (100 trees) | Sizey baseline 4 |
| **LGBM** | LightGBM (200 trees) | Our preferred |
| **NGB** | NGBoost LogNormal | Distributional, used by AL gates |

All models train twice per bucket: **Sizey** (X = `[log a]`) and **Joint** (X = `[log a, log c]`,
only on c-present rows). 12 model fits per bucket × 14 buckets = 168 base fits per CV fold.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from lightgbm import LGBMRegressor
try:
    from ngboost import NGBRegressor
    from ngboost.distns import LogNormal
    HAS_NGB = True
except ImportError:
    HAS_NGB = False

def lr_():   return LinearRegression()
def knn_():  return Pipeline([("s",StandardScaler()),("m",KNeighborsRegressor(n_neighbors=5))])
def mlp_():  return Pipeline([("s",StandardScaler()),
                              ("m",MLPRegressor(hidden_layer_sizes=(64,32),
                                                max_iter=400, random_state=SEED))])
def rf_():   return RandomForestRegressor(n_estimators=100, random_state=SEED, n_jobs=1)
def lgbm_(): return LGBMRegressor(n_estimators=200, learning_rate=0.05, num_leaves=31,
                                  min_data_in_leaf=2, verbose=-1, random_state=SEED)
def ngb_():
    if not HAS_NGB: return None
    return NGBRegressor(Dist=LogNormal, n_estimators=200, learning_rate=0.05,
                        verbose=False, random_state=SEED)

MODELS = {"lr":lr_, "knn":knn_, "mlp":mlp_, "rf":rf_, "lgbm":lgbm_, "ngb":ngb_}
print(f"NGBoost available: {HAS_NGB}")

## 5.  Train all models — Sizey + Joint feature variants

In [ ]:
def cv_folds(n, k=5, seed=SEED):
    idx = np.random.default_rng(seed).permutation(n)
    chunks = np.array_split(idx, min(k, n))
    for i, te in enumerate(chunks):
        tr = np.concatenate([c for j,c in enumerate(chunks) if j != i])
        if len(tr) and len(te): yield tr, te

def fit_predict(model_fn, X_tr, y_tr, X_te, log_y=True):
    m = model_fn()
    if m is None: return None
    if isinstance(m, (NGBRegressor,) if HAS_NGB else ()):
        m.fit(X_tr, np.exp(y_tr))
        return np.log(np.maximum(1.0, m.pred_dist(X_te).mean()))
    m.fit(X_tr, y_tr)
    return m.predict(X_te)

# Train per bucket; for each model, do an inner 5-fold CV on TRAIN to calibrate σ,
# then refit on full TRAIN, predict TEST.
results = []      # per-row test predictions
trained = {}      # for AL + live-prediction cells later
t0 = time.time()
for (wf, proc), (tr_df, te_df) in splits.items():
    a_tr = np.log(tr_df["a"].values + EPS); M_tr = np.log(tr_df["M"].values + EPS)
    a_te = np.log(te_df["a"].values + EPS); M_te_raw = te_df["M"].values
    rt_te = te_df["runtime"].values
    c_tr_mask = tr_df["c"].notna().values
    c_te_mask = te_df["c"].notna().values

    bucket_record = {"wf":wf, "proc":proc, "n_tr":len(tr_df), "n_te":len(te_df),
                     "n_tr_c":int(c_tr_mask.sum()), "n_te_c":int(c_te_mask.sum()),
                     "M_te":M_te_raw, "rt_te":rt_te, "c_te_mask":c_te_mask,
                     "a_te_log":a_te,
                     "preds":{}, "sigmas":{}, "fits":{}}

    for name, fn in MODELS.items():
        # ---- Sizey (a only) ----
        try:
            # Inner CV → σ
            p_cv = np.full(len(tr_df), np.nan)
            for tr_i, te_i in cv_folds(len(tr_df)):
                p_cv[te_i] = fit_predict(fn, a_tr[tr_i].reshape(-1,1), M_tr[tr_i],
                                         a_tr[te_i].reshape(-1,1))
            σ_s = float(np.nanstd(M_tr - p_cv))
            # Refit on full train, predict test
            full_m = fn(); full_m.fit(a_tr.reshape(-1,1),
                                      np.exp(M_tr) if isinstance(full_m, (NGBRegressor,) if HAS_NGB else ()) else M_tr)
            if isinstance(full_m, (NGBRegressor,) if HAS_NGB else ()):
                p_te = np.log(np.maximum(1.0, full_m.pred_dist(a_te.reshape(-1,1)).mean()))
            else:
                p_te = full_m.predict(a_te.reshape(-1,1))
            bucket_record["preds"][f"{name}_sizey"] = p_te
            bucket_record["sigmas"][f"{name}_sizey"] = σ_s
            bucket_record["fits"][f"{name}_sizey"] = full_m
        except Exception as e:
            bucket_record["preds"][f"{name}_sizey"] = None

        # ---- Joint (a + c) ----
        if c_tr_mask.sum() < MIN_INSTANCES or c_te_mask.sum() == 0:
            continue
        try:
            a_tr_c = a_tr[c_tr_mask]; c_tr_c = np.log(tr_df["c"].values[c_tr_mask] + EPS)
            M_tr_c = M_tr[c_tr_mask]
            X_tr_full = np.column_stack([a_tr_c, c_tr_c])
            p_cv_j = np.full(len(a_tr_c), np.nan)
            for tr_i, te_i in cv_folds(len(a_tr_c)):
                p_cv_j[te_i] = fit_predict(fn, X_tr_full[tr_i], M_tr_c[tr_i], X_tr_full[te_i])
            σ_j = float(np.nanstd(M_tr_c - p_cv_j))
            full_m = fn()
            if isinstance(full_m, (NGBRegressor,) if HAS_NGB else ()):
                full_m.fit(X_tr_full, np.exp(M_tr_c))
            else:
                full_m.fit(X_tr_full, M_tr_c)
            a_te_c = a_te[c_te_mask]; c_te_c = np.log(te_df["c"].values[c_te_mask] + EPS)
            X_te_c = np.column_stack([a_te_c, c_te_c])
            if isinstance(full_m, (NGBRegressor,) if HAS_NGB else ()):
                p_te_c = np.log(np.maximum(1.0, full_m.pred_dist(X_te_c).mean()))
            else:
                p_te_c = full_m.predict(X_te_c)
            # Place predictions back into the bucket-aligned array (NaN where c missing)
            p_te_full = np.full(len(te_df), np.nan); p_te_full[c_te_mask] = p_te_c
            bucket_record["preds"][f"{name}_joint"] = p_te_full
            bucket_record["sigmas"][f"{name}_joint"] = σ_j
            bucket_record["fits"][f"{name}_joint"] = full_m
        except Exception as e:
            bucket_record["preds"][f"{name}_joint"] = None

    results.append(bucket_record)
    trained[f"{wf}::{proc}"] = bucket_record
    print(f"  {wf}::{proc[:50]:<50}  done")
print(f"\nTotal training time: {time.time()-t0:.1f}s")

## 6.  Aggregate held-out evaluation per model

In [ ]:
def evaluate(results, key):
    """Return aggregate metrics across all buckets for a given model+feature key."""
    all_pred, all_M, all_rt, all_sigma_per_row = [], [], [], []
    for r in results:
        p = r["preds"].get(key)
        if p is None: continue
        mask = ~np.isnan(p)
        if not mask.any(): continue
        σ = r["sigmas"].get(key, 0.5)
        all_pred.append(p[mask]); all_M.append(r["M_te"][mask])
        all_rt.append(r["rt_te"][mask]); all_sigma_per_row.append(np.full(mask.sum(), σ))
    if not all_pred: return None
    pred = np.concatenate(all_pred); M = np.concatenate(all_M)
    rt = np.concatenate(all_rt); σs = np.concatenate(all_sigma_per_row)
    safe = np.exp(pred + K * σs)
    return dict(
        n      = len(pred),
        MAPE   = float(np.mean(np.abs(np.exp(pred) - M) / M) * 100),
        R2     = float(1 - np.sum((np.log(M)-pred)**2) / np.sum((np.log(M)-np.log(M).mean())**2)),
        cov2x  = float(np.mean((np.exp(pred) >= 0.5*M) & (np.exp(pred) <= 2.0*M)) * 100),
        wastage_GBh = float(np.sum(np.maximum(safe - M, 0)/1024**3 * rt/3600)),
        OOM    = int(np.sum(safe < M)),
    )

rows = []
for name in MODELS:
    for variant in ["sizey", "joint"]:
        e = evaluate(results, f"{name}_{variant}")
        if e is None: continue
        rows.append({"model":name, "feature":variant, **e})
metrics = pd.DataFrame(rows)
print(metrics.to_string(index=False))

## 7.  FIGURE — MAPE by model: Sizey vs Joint

In [ ]:
fig, ax = plt.subplots(figsize=(13, 7))
order = ["lr","knn","mlp","rf","lgbm","ngb"]
sizey_vals = [metrics[(metrics.model==m)&(metrics.feature=="sizey")].MAPE.iloc[0] for m in order]
joint_vals = [metrics[(metrics.model==m)&(metrics.feature=="joint")].MAPE.iloc[0] for m in order]
x = np.arange(len(order)); w = 0.38
b1 = ax.bar(x-w/2, sizey_vals, w, label="Sizey  (a only)", color=COLORS["sizey"], edgecolor="black", linewidth=1.5)
b2 = ax.bar(x+w/2, joint_vals, w, label="Joint  (a, c)",   color=COLORS["joint"], edgecolor="black", linewidth=1.5)
for b, v in zip(b1, sizey_vals): ax.text(b.get_x()+b.get_width()/2, v+0.3, f"{v:.1f}", ha="center", fontsize=15)
for b, v in zip(b2, joint_vals): ax.text(b.get_x()+b.get_width()/2, v+0.3, f"{v:.1f}", ha="center", fontsize=15, fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels([m.upper() for m in order])
ax.set_ylabel("Held-out MAPE (%)"); ax.set_title("Memory prediction error: Sizey vs Joint per model class")
ax.legend(loc="upper left", frameon=True)
ax.grid(axis="y", alpha=0.4, linestyle="--")
fig.savefig(FIG_DIR/"fig2_mape_by_model.png"); plt.show()

## 8.  FIGURE — Predicted vs actual M, calibration scatter (LightGBM Joint)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 7))
for ax, key, title in zip(axes, ["lgbm_sizey", "lgbm_joint"],
                          ["LGBM Sizey  (a only)", "LGBM Joint  (a + c)"]):
    all_p, all_M = [], []
    for r in results:
        p = r["preds"].get(key)
        if p is None: continue
        mask = ~np.isnan(p)
        all_p.append(np.exp(p[mask])); all_M.append(r["M_te"][mask])
    P = np.concatenate(all_p); M = np.concatenate(all_M)
    ax.scatter(M/1024**2, P/1024**2, alpha=0.35, s=70, edgecolor="white", linewidth=0.5,
               color=COLORS["sizey"] if "sizey" in key else COLORS["joint"])
    lo, hi = min(M.min(), P.min())/1024**2, max(M.max(), P.max())/1024**2
    ax.plot([lo, hi], [lo, hi], "k--", linewidth=2.5, label="perfect")
    ax.plot([lo, hi], [2*lo, 2*hi], "r:", linewidth=2, alpha=0.7, label="2× over")
    ax.plot([lo, hi], [0.5*lo, 0.5*hi], "r:", linewidth=2, alpha=0.7, label="2× under")
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("Actual peak RSS (MB)"); ax.set_ylabel("Predicted M (MB)")
    ax.set_title(title); ax.legend(loc="lower right")
    ax.grid(True, which="both", alpha=0.3)
fig.suptitle("Predicted vs actual peak RSS — held-out set")
fig.tight_layout()
fig.savefig(FIG_DIR/"fig3_calibration.png"); plt.show()

## 9.  FIGURE — Per-bucket coverage and σ

In [ ]:
# Coverage = % of held-out rows where safe-alloc >= true M
fig, ax = plt.subplots(figsize=(13, 7))
bucket_labels, sizey_cov, joint_cov = [], [], []
for r in results:
    bucket_labels.append(f"{r['wf']}::{r['proc'][:30]}")
    p_s = r["preds"].get("lgbm_sizey")
    σ_s = r["sigmas"].get("lgbm_sizey", 0.5)
    if p_s is not None:
        m = ~np.isnan(p_s)
        cov = np.mean(np.exp(p_s[m] + K*σ_s) >= r["M_te"][m]) * 100
        sizey_cov.append(cov)
    else: sizey_cov.append(np.nan)
    p_j = r["preds"].get("lgbm_joint")
    σ_j = r["sigmas"].get("lgbm_joint", None)
    if p_j is not None and σ_j is not None:
        m = ~np.isnan(p_j)
        cov = np.mean(np.exp(p_j[m] + K*σ_j) >= r["M_te"][m]) * 100
        joint_cov.append(cov)
    else: joint_cov.append(np.nan)

y = np.arange(len(bucket_labels))
ax.scatter(sizey_cov, y, s=320, marker="o", color=COLORS["sizey"], edgecolor="black",
           linewidth=2, label="Sizey", zorder=3)
ax.scatter(joint_cov, y, s=320, marker="D", color=COLORS["joint"], edgecolor="black",
           linewidth=2, label="Joint", zorder=4)
ax.axvline(95, linestyle="--", color="gray", alpha=0.6, linewidth=2)
ax.set_yticks(y); ax.set_yticklabels(bucket_labels, fontsize=13)
ax.set_xlabel("Coverage on held-out test (%)"); ax.set_xlim(50, 102)
ax.set_title("Per-bucket coverage  (safe_alloc ≥ M_actual)")
ax.legend(loc="lower right"); ax.invert_yaxis()
ax.grid(axis="x", alpha=0.4, linestyle="--")
fig.savefig(FIG_DIR/"fig4_coverage.png"); plt.show()

## 10.  Active-Learning gates

We compute three audit-priority scores per held-out row:
- **G1 (variance)**  — bucket-level σ from NGBoost (or Sizey σ as proxy)
- **G2 (risk)**       — `P(M > 90th-pct(M_train))` from a Gaussian on (μ, σ)
- **G3 (disagree)**   — `|c_predicted − c_actual|` from a per-bucket LR(a → c)

Combined gate = top-(B/3) union from each. We sweep budget `B ∈ {0.05, 0.10, 0.25, 0.50, 1.00}`.

In [ ]:
from scipy.stats import norm as _norm
BUDGETS = [0.05, 0.10, 0.25, 0.50, 1.00]
GATES   = ["random","g1_variance","g2_risk","g3_disagree","combined"]

def score_gate(r, gate, rng):
    p_s = r["preds"].get("lgbm_sizey"); σ_s = r["sigmas"].get("lgbm_sizey", 0.5)
    n = len(p_s) if p_s is not None else 0
    if gate == "random":
        return rng.random(n)
    if gate == "g1_variance":
        return np.full(n, σ_s)  # bucket-level σ → constant per bucket; useful across buckets
    if gate == "g2_risk":
        # P(M > capacity) where capacity = 90th pct of train M for this bucket
        wf, proc = r["wf"], r["proc"]
        tr_df = splits[(wf, proc)][0]
        cap_log = np.log(np.percentile(tr_df["M"].values, 90) + EPS)
        return 1 - _norm.cdf((cap_log - p_s) / max(σ_s, 1e-3))
    if gate == "g3_disagree":
        wf, proc = r["wf"], r["proc"]
        tr_df = splits[(wf, proc)][0]
        c_mask_tr = tr_df["c"].notna().values
        if c_mask_tr.sum() < 2:
            return np.zeros(n)
        from sklearn.linear_model import LinearRegression as LR
        a_tr_c = np.log(tr_df["a"].values[c_mask_tr]+EPS).reshape(-1,1)
        c_tr_c = np.log(tr_df["c"].values[c_mask_tr]+EPS)
        lr = LR().fit(a_tr_c, c_tr_c)
        # apply to test rows where we know actual c
        a_te = r["a_te_log"].reshape(-1,1)
        c_pred = lr.predict(a_te)
        c_te_act = np.full(n, np.nan)
        te_df = splits[(wf, proc)][1]
        c_te_act[r["c_te_mask"]] = np.log(te_df["c"].values[r["c_te_mask"]]+EPS)
        return np.where(np.isnan(c_te_act), 0.0, np.abs(c_pred - c_te_act))
    return None

def select_audits(scores, B, gate, rng):
    n = len(scores); k = max(1, int(np.ceil(n*B)))
    if gate == "combined":
        sets = [set(np.argsort(score_gate(r, g, rng))[-max(1,k//3):].tolist())
                for r,g in zip([_R]*3, ["g1_variance","g2_risk","g3_disagree"])]
        idx = list(set().union(*sets))[:k]
    else:
        idx = np.argsort(scores)[-k:]
    audit = np.zeros(n, dtype=bool); audit[idx] = True
    return audit

# Sweep
al_rows = []
for B in BUDGETS:
    for gate in GATES:
        rng = np.random.default_rng(7)
        total_w, total_f, total_n = 0.0, 0, 0
        for r in results:
            n = len(r["M_te"])
            if gate == "random":
                k = max(1, int(np.ceil(n*B)))
                audit = np.zeros(n, dtype=bool)
                audit[rng.choice(n, k, replace=False)] = True
            elif gate == "combined":
                _R = r
                idx_set = set()
                for g in ["g1_variance","g2_risk","g3_disagree"]:
                    s = score_gate(r, g, rng)
                    k = max(1, int(np.ceil(n*B/3)))
                    idx_set.update(np.argsort(s)[-k:].tolist())
                audit = np.zeros(n, dtype=bool)
                idx_arr = np.array(list(idx_set)[:max(1,int(np.ceil(n*B)))])
                audit[idx_arr] = True
            else:
                s = score_gate(r, gate, rng)
                k = max(1, int(np.ceil(n*B)))
                audit = np.zeros(n, dtype=bool)
                audit[np.argsort(s)[-k:]] = True

            # Audited rows: use joint pred if available, else fall back to sizey
            p_s = r["preds"].get("lgbm_sizey"); σ_s = r["sigmas"].get("lgbm_sizey", 0.5)
            p_j = r["preds"].get("lgbm_joint"); σ_j = r["sigmas"].get("lgbm_joint", σ_s)
            if p_s is None: continue
            pred = np.where(audit & (p_j is not None) & (~np.isnan(p_j) if p_j is not None else False),
                            p_j if p_j is not None else p_s, p_s)
            σ_per_row = np.where(audit & (p_j is not None) &
                                  (~np.isnan(p_j) if p_j is not None else False),
                                  σ_j, σ_s)
            safe = np.exp(pred + K*σ_per_row)
            M = r["M_te"]; rt = r["rt_te"]
            total_w += float(np.sum(np.maximum(safe - M, 0)/1024**3 * rt/3600))
            total_f += int(np.sum(safe < M))
            total_n += n
        al_rows.append({"budget":B, "gate":gate, "n":total_n,
                        "wastage_GBh":total_w, "failures":total_f})

al = pd.DataFrame(al_rows)
print(al.pivot(index="budget", columns="gate", values="wastage_GBh").round(3))
print()
print(al.pivot(index="budget", columns="gate", values="failures"))

## 11.  FIGURE — AL budget × gate sweep

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
gate_colors = {"random":"#7f7f7f","g1_variance":"#1f77b4","g2_risk":"#d62728",
               "g3_disagree":"#2ca02c","combined":"#9467bd"}
markers = {"random":"o","g1_variance":"s","g2_risk":"D","g3_disagree":"^","combined":"P"}
for metric, ax, ylab in [("wastage_GBh", axes[0], "Total wastage (GB·h)"),
                          ("failures",    axes[1], "Total OOMs")]:
    for g in GATES:
        sub = al[al.gate==g].sort_values("budget")
        ax.plot(sub.budget, sub[metric], marker=markers[g], color=gate_colors[g],
                label=g, markeredgecolor="black", markeredgewidth=1.5)
    ax.set_xlabel("Audit budget B"); ax.set_ylabel(ylab)
    ax.set_xscale("log"); ax.grid(True, which="both", alpha=0.4)
    ax.legend(loc="best", fontsize=14)
fig.suptitle("Active-learning sweep — wastage and OOM vs audit budget")
fig.tight_layout()
fig.savefig(FIG_DIR/"fig5_al_sweep.png"); plt.show()

## 12.  Save artifacts

In [ ]:
MD = REPO/"models_unified"; MD.mkdir(exist_ok=True)
to_save = {}
for k, r in trained.items():
    to_save[k] = {
        "fits":   {fn: r["fits"].get(fn) for fn in r["fits"]},
        "sigmas": r["sigmas"],
        "n_tr":   r["n_tr"], "n_te":   r["n_te"],
        "n_tr_c": r["n_tr_c"], "n_te_c": r["n_te_c"],
        "safety_k": K,
    }
PKL = MD/"per_task_paper.pkl"
with open(PKL, "wb") as f: pickle.dump(to_save, f)
print(f"wrote {PKL}  ({PKL.stat().st_size:,} bytes)")

## 13.  Live predictions on held-out tasks

In [ ]:
# Pick one row per bucket from the held-out test set; show predictions for all 6 models × 2 variants.
import random
random.seed(99)
sample_keys = list(trained.keys())[:6]
demo_rows = []
for k in sample_keys:
    r = trained[k]
    if r["n_te"] == 0: continue
    te_df = splits[tuple(k.split('::',1))][1]
    pick = te_df.sample(1, random_state=11).iloc[0]
    a, c, M = pick["a"], pick["c"], pick["M"]
    row = {"bucket":k[:50], "a_MB":a/1024**2, "c_MB":(c/1024**2 if not np.isnan(c) else None), "M_MB":M/1024**2}
    for name in MODELS:
        for variant in ["sizey","joint"]:
            p_arr = r["preds"].get(f"{name}_{variant}")
            if p_arr is None: row[f"{name}_{variant}"] = None; continue
            σ = r["sigmas"].get(f"{name}_{variant}", 0.5)
            # the pick is in te_df; find its index in the held-out array
            te_idx = te_df.index.get_loc(pick.name)
            v = p_arr[te_idx]
            if np.isnan(v): row[f"{name}_{variant}"] = None
            else: row[f"{name}_{variant}"] = float(np.exp(v + K*σ)) / 1024**2  # MB
    demo_rows.append(row)
demo = pd.DataFrame(demo_rows)
demo.round(1)

## 14.  Sizey ensemble (per-bucket best-of-4 baseline)

The original Sizey paper trains an ensemble of 4 base models (LR, KNN, MLP, RF) and selects
the **best one per bucket** at training time. This is the published baseline we compare against.
Below we replicate the Sizey selection logic on our held-out set and contrast against our methods.

In [ ]:
# For each bucket, pick the Sizey baseline (LR/KNN/MLP/RF) with lowest train-CV MAPE
# (we approximate with held-out MAPE here, computed PER BUCKET).
sizey_choice = {}     # bucket -> (best_model_name, MAPE)
sizey_per_row_pred  = []
sizey_per_row_M     = []
sizey_per_row_rt    = []
sizey_per_row_sigma = []

for r in results:
    bk = f"{r['wf']}::{r['proc']}"
    best_name, best_mape, best_pred, best_sigma = None, np.inf, None, None
    for cand in ["lr","knn","mlp","rf"]:
        p = r["preds"].get(f"{cand}_sizey")
        if p is None: continue
        mask = ~np.isnan(p)
        if not mask.any(): continue
        mape = float(np.mean(np.abs(np.exp(p[mask]) - r["M_te"][mask]) / r["M_te"][mask]) * 100)
        if mape < best_mape:
            best_mape, best_name, best_pred = mape, cand, p
            best_sigma = r["sigmas"].get(f"{cand}_sizey", 0.5)
    if best_name is None: continue
    sizey_choice[bk] = (best_name, best_mape)
    mask = ~np.isnan(best_pred)
    sizey_per_row_pred.append(best_pred[mask])
    sizey_per_row_M.append(r["M_te"][mask])
    sizey_per_row_rt.append(r["rt_te"][mask])
    sizey_per_row_sigma.append(np.full(mask.sum(), best_sigma))

print("Per-bucket Sizey selection (best of LR/KNN/MLP/RF):")
for bk, (name, mape) in sizey_choice.items():
    print(f"  {bk[:60]:<60}  -> {name.upper():>4}  (MAPE {mape:.1f}%)")

In [ ]:
# Aggregate Sizey ensemble metrics
sizey_pred  = np.concatenate(sizey_per_row_pred)
sizey_M     = np.concatenate(sizey_per_row_M)
sizey_rt    = np.concatenate(sizey_per_row_rt)
sizey_sigma = np.concatenate(sizey_per_row_sigma)
sizey_safe  = np.exp(sizey_pred + K * sizey_sigma)

sizey_ens = dict(
    n=len(sizey_pred),
    MAPE=float(np.mean(np.abs(np.exp(sizey_pred) - sizey_M) / sizey_M) * 100),
    R2=float(1 - np.sum((np.log(sizey_M)-sizey_pred)**2) / np.sum((np.log(sizey_M)-np.log(sizey_M).mean())**2)),
    cov2x=float(np.mean((np.exp(sizey_pred) >= 0.5*sizey_M) & (np.exp(sizey_pred) <= 2.0*sizey_M)) * 100),
    wastage_GBh=float(np.sum(np.maximum(sizey_safe - sizey_M, 0)/1024**3 * sizey_rt/3600)),
    OOM=int(np.sum(sizey_safe < sizey_M)),
)
joint_lgbm = metrics[(metrics.model=="lgbm")&(metrics.feature=="joint")].iloc[0].to_dict()
joint_ngb  = metrics[(metrics.model=="ngb")&(metrics.feature=="joint")].iloc[0].to_dict() if "ngb" in metrics.model.values else None

print("\n=== HEAD-TO-HEAD vs Sizey ensemble (paper's published baseline) ===\n")
hdr = "{:<28}{:>10}{:>10}{:>10}{:>14}{:>10}".format("method","MAPE %","R²","cov 2× %","wastage GBh","OOMs")
print(hdr); print("-"*len(hdr))
print("{:<28}{:>10.2f}{:>10.3f}{:>10.1f}{:>14.3f}{:>10}".format(
    "Sizey ensemble (best/bucket)", sizey_ens["MAPE"], sizey_ens["R2"],
    sizey_ens["cov2x"], sizey_ens["wastage_GBh"], sizey_ens["OOM"]))
print("{:<28}{:>10.2f}{:>10.3f}{:>10.1f}{:>14.3f}{:>10}".format(
    "Ours: LGBM Joint", joint_lgbm["MAPE"], joint_lgbm["R2"],
    joint_lgbm["cov2x"], joint_lgbm["wastage_GBh"], joint_lgbm["OOM"]))
if joint_ngb is not None:
    print("{:<28}{:>10.2f}{:>10.3f}{:>10.1f}{:>14.3f}{:>10}".format(
        "Ours: NGB Joint", joint_ngb["MAPE"], joint_ngb["R2"],
        joint_ngb["cov2x"], joint_ngb["wastage_GBh"], joint_ngb["OOM"]))
print()
print(f"Joint LGBM vs Sizey ensemble:")
print(f"  MAPE     {sizey_ens['MAPE']:.2f}% -> {joint_lgbm['MAPE']:.2f}%   ({(1-joint_lgbm['MAPE']/sizey_ens['MAPE'])*100:.1f}% lower error)")
print(f"  Wastage  {sizey_ens['wastage_GBh']:.3f} GBh -> {joint_lgbm['wastage_GBh']:.3f} GBh  ({(1-joint_lgbm['wastage_GBh']/max(sizey_ens['wastage_GBh'],1e-9))*100:.1f}% cut)")
print(f"  OOMs     {sizey_ens['OOM']} -> {joint_lgbm['OOM']}   ({(1-joint_lgbm['OOM']/max(sizey_ens['OOM'],1e-9))*100:.1f}% cut)")

## 15.  FIGURE — Final head-to-head vs Sizey ensemble

In [ ]:
# Bar chart of all 4 metrics: Sizey ensemble vs our LGBM Joint vs our NGB Joint
fig, axes = plt.subplots(1, 4, figsize=(20, 6))
metric_keys = [("MAPE","Held-out MAPE (%)"), ("R2","R² (log)"),
               ("wastage_GBh","Wastage (GB·h)"), ("OOM","OOMs")]
methods = [("Sizey\nensemble", sizey_ens, "#6c757d"),
           ("Ours\nLGBM Joint", joint_lgbm, "#16a085")]
if joint_ngb is not None:
    methods.append(("Ours\nNGB Joint", joint_ngb, "#a65628"))

for ax, (key, label) in zip(axes, metric_keys):
    vals = [m[1][key] for m in methods]
    bars = ax.bar(range(len(methods)), vals,
                  color=[m[2] for m in methods], edgecolor="black", linewidth=2)
    for b, v in zip(bars, vals):
        h = b.get_height()
        ax.text(b.get_x()+b.get_width()/2, h*1.02 if h>0 else 0,
                f"{v:.2f}" if key in ("MAPE","R2","wastage_GBh") else f"{int(v)}",
                ha="center", fontsize=16, fontweight="bold")
    ax.set_xticks(range(len(methods))); ax.set_xticklabels([m[0] for m in methods], fontsize=14)
    ax.set_ylabel(label); ax.grid(axis="y", alpha=0.4, linestyle="--")
    if key == "MAPE": ax.set_ylim(0, max(vals)*1.25)
fig.suptitle("Sizey ensemble (paper baseline) vs our c-augmented joint models")
fig.tight_layout()
fig.savefig(FIG_DIR/"fig6_sizey_vs_ours.png"); plt.show()

## 16.  Summary

In [ ]:
print("=" * 80)
print("FINAL SUMMARY  (held-out test, 6,735 rows / 14 buckets)")
print("=" * 80)
sizey_lgbm = metrics[(metrics.model=="lgbm")&(metrics.feature=="sizey")].iloc[0]
joint_lgbm = metrics[(metrics.model=="lgbm")&(metrics.feature=="joint")].iloc[0]
print(f"  LGBM Sizey:  MAPE={sizey_lgbm.MAPE:.2f}%  R²={sizey_lgbm.R2:.3f}  "
      f"cov(2x)={sizey_lgbm.cov2x:.1f}%  wastage={sizey_lgbm.wastage_GBh:.3f} GBh  OOMs={sizey_lgbm.OOM}")
print(f"  LGBM Joint:  MAPE={joint_lgbm.MAPE:.2f}%  R²={joint_lgbm.R2:.3f}  "
      f"cov(2x)={joint_lgbm.cov2x:.1f}%  wastage={joint_lgbm.wastage_GBh:.3f} GBh  OOMs={joint_lgbm.OOM}")
print()
print(f"  Joint vs Sizey (LightGBM):")
print(f"    MAPE drops by {(1-joint_lgbm.MAPE/sizey_lgbm.MAPE)*100:.1f}%")
print(f"    Wastage drops by {(1-joint_lgbm.wastage_GBh/sizey_lgbm.wastage_GBh)*100:.1f}%")
print(f"    OOMs drop by {(1-joint_lgbm.OOM/sizey_lgbm.OOM)*100:.1f}%")
print()
print(f"  Best AL gate at B=0.10:  ", end="")
b10 = al[al.budget==0.10].sort_values("wastage_GBh").iloc[0]
print(f"{b10.gate}  (wastage={b10.wastage_GBh:.3f} GBh)")
print(f"  Audit-everything (B=1.0): ", end="")
b1 = al[(al.budget==1.0)&(al.gate=="random")].iloc[0]
print(f"wastage={b1.wastage_GBh:.3f} GBh")
print()
print("Figures saved to:", FIG_DIR)